# Run the GPT-2 experiment

This notebook asks whether a feature inside GPT-2 tracks which name came first. Run the cells from top to bottom. You need PyTorch and access to the GPT-2 weights. A GPU makes the experiment faster. In Colab, choose a GPU under **Runtime → Change runtime type**.

To check the saved results on a CPU, open the [results notebook](inspect_results.ipynb).

## Open the project

On your computer, unzip the project and start Jupyter in that folder. In a new Colab session, run the next cell and choose the project ZIP. You can get the ZIP from **Code → Download ZIP** on GitHub.

In [ ]:
import os
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

here = Path.cwd().resolve()
root = next((p for p in (here, *here.parents) if (p / "run_pipeline.py").is_file()), None)
if root is None:
    from google.colab import files
    uploaded = files.upload()
    archives = [Path(name) for name in uploaded if name.lower().endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Select one ZIP of the project source.")
    extracted = Path(tempfile.mkdtemp(prefix="ioi-project-")).resolve()
    with zipfile.ZipFile(archives[0]) as archive:
        if any(not (extracted / item.filename).resolve().is_relative_to(extracted)
               for item in archive.infolist()):
            raise ValueError("Unexpected path in the ZIP.")
        archive.extractall(extracted)
    candidates = [p.parent for p in extracted.rglob("run_pipeline.py") if (p.parent / "src").is_dir()]
    if len(candidates) != 1:
        raise ValueError("The ZIP must contain a single IOI project.")
    root = candidates[0]
os.chdir(root)
print("Project directory:", root)

## Install the packages and test the code

The tests use small examples. They check how the code makes prompts, saves GPT-2 values, removes features, and records the results.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
import torch
print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## Make the prompts

Each prompt has two names. The recipient comes first in half the prompts. Training, development, and test prompts use different names. The OOD set uses the test names in two new sentence patterns. This notebook saves each run in a new folder inside `runs/`.

In [ ]:
from datetime import datetime, timezone
run_dir = root / "runs" / datetime.now(timezone.utc).strftime("notebook-%Y%m%d-%H%M%S-%f")
run_dir.mkdir(parents=True, exist_ok=False)

def run_stage(stage):
    subprocess.run(
        [sys.executable, "run_pipeline.py", "--stage", stage, "--out", str(run_dir), "--seed", "42"],
        check=True,
    )

run_stage("data")

## Check GPT-2 before changing it

For each development prompt, compare the scores of the recipient and the subject. The gap is positive when GPT-2 gives the recipient a higher score.

In [ ]:
run_stage("baseline")

## Train the sparse autoencoder (SAE)

Save the GPT-2 values at the last token, after block 8. The first block is block 0. The SAE learns to rebuild these values using up to 32 of its 1,536 features for each prompt. Keep the saved model with the smallest error on the development set.

In [ ]:
run_stage("train")

## Pick a feature and remove it

Use the development set to pick a feature that tracks name order. Remove it and measure the change in the score gap. Repeat this step for other features so you can compare them in your own run.

In [ ]:
run_stage("evaluate")

## Look at the results

These scores come from the run you just finished. If you train a new SAE, the feature number may change.

In [ ]:
import json
import matplotlib.pyplot as plt

with (run_dir / "summary.json").open() as f:
    summary = json.load(f)
print("Selected feature:", summary["selected_feature_id"])
for split in ("test", "ood"):
    s = summary[split]
    print(split, "order AUROC", s["feature_order_auc"],
          "average change in the gap", s["selected_feature"]["mean_gap_change"])
for filename in ("sae_loss.png", "causal_controls.png"):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.imshow(plt.imread(run_dir / filename))
    ax.axis("off")
    plt.show()

## Save the files

This cell puts the prompts, scores, GPT-2 values, and SAE weights in a ZIP. You can find it next to your run folder. In Colab, you can download it.

In [ ]:
import shutil
archive_path = shutil.make_archive(str(run_dir), "zip", run_dir)
print(archive_path)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(archive_path)